# Day 38: Music & Audio Generation with AudioLDM 2

Generate short audio clips from text prompts.

In [ ]:
import torch
from diffusers import AudioLDM2Pipeline
import scipy.io.wavfile as wavfile
import numpy as np
from IPython.display import Audio

In [ ]:
# Load pipeline (may take a few minutes, downloads ~1GB)
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipe = pipe.to(device)
print("Pipeline loaded")

In [ ]:
# Generate a short melody
prompt = "lo-fi beat with piano and soft drums, relaxing"
negative_prompt = "distorted, noisy, high tempo"

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_length_in_s=5.0,
    guidance_scale=2.5,
).audios[0]

print(f"Generated audio shape: {audio.shape}, sample rate: {pipe.vae.sampling_rate}")
# Audio is a numpy array of shape (channels, samples) – usually mono so shape (1, samples)

In [ ]:
# Save to file and play
sampling_rate = pipe.vae.sampling_rate
wavfile.write("generated_music.wav", sampling_rate, (audio[0] * 32767).astype(np.int16))
Audio("generated_music.wav")

In [ ]:
# Try a sound effect
sfx_prompt = "thunderstorm with heavy rain"
audio_sfx = pipe(sfx_prompt, audio_length_in_s=3.0, num_inference_steps=150).audios[0]
wavfile.write("thunderstorm.wav", sampling_rate, (audio_sfx[0] * 32767).astype(np.int16))
Audio("thunderstorm.wav")

In [ ]:
# Experiment with duration
for duration in [3.0, 5.0, 8.0]:
    audio = pipe("calm piano melody", audio_length_in_s=duration, num_inference_steps=150).audios[0]
    wavfile.write(f"piano_{duration}s.wav", sampling_rate, (audio[0] * 32767).astype(np.int16))
    print(f"Generated {duration} seconds")